
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



# 데모 - 회수 에이전트 구축 및 기록

## 개요

이번 데모에서는 Databricks Mosaic AI를 사용해 운영 준비가 가능한 검색 에이전트를 만들고 로그하는 방법을 살펴보겠습니다. 검색 에이전트는 대규모 언어 모델의 힘과 조직의 지식 기반을 결합하여 정확하고 맥락 인식 있는 응답을 제공합니다. AI Playground에서 AI Search 테스트 과정을 진행하고, LangChain로 에이전트를 구축하며, MLflow 관찰 추적 구현, 배포용 모델로 에이전트를 등록하는 과정을 살펴보겠습니다.

## 학습 목표
이 데모가 끝날 때쯤이면 다음을 할 수 있게 됩니다:
- AI Search 기능을 AI Playground UI를 사용하여 빠른 프로토타이핑을 위해 **테스트**합니다.
- LangChain AI Search 인덱스를 사용하여 검색 에이전트를 **구축**합니다.
- 에이전트 상호작용을 모니터링하고 디버그하기 위해 MLflow 추적을 **구현**하세요.
- 에이전트를 Model Registry에 모델로 **등록**하십시오.

## 요구 사항:
- 미리 생성된 **AI Search 엔드포인트**. 이것은 미리 생성되었습니다.
- **서버리스 Compute (환경 버전 5)**. [여기](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version)에 따라 적절한 환경 버전을 선택하세요.
- LangChain 및 회수 증강 생성(RAG) 개념에 대한 기본적인 이해.

## 준비

아래 코드를 실행하여 필요한 라이브러리를 설치하고 교실 환경을 구성하세요. 이 단계는 모든 의존성이 사용 가능하고 워크스페이스가 데모 준비가 완료되도록 보장합니다.

In [0]:
%run ../Includes/Classroom-Setup-04 $section="demo"

## A. AI Playground에서 AI Search 테스트

검색 에이전트를 프로그래매틱하게 만들기 전에 **AI Playground**를 사용해 AI Search 인덱스를 테스트할 것입니다. AI Playground는 AI Search 인덱스를 **실험할 수 있는 사용자 친화적인 인터페이스**를 제공하여, 지식 기반이 올바르게 작동하는지 빠르게 검증하고 검색 시스템이 다양한 쿼리에 어떻게 반응하는지 이해할 수 있습니다.

이 인터랙티브 테스트 단계는 검색 품질을 이해하고, 임베딩이나 청킹 전략의 잠재적 문제를 파악하며, 코드 개발에 시간을 투자하기 전에 접근 방식을 개선하는 데 유용합니다.

**데이터셋 정보:** AI Search 인덱스에는 Orion이라는 이름의 로봇을 위한 가상의 로봇 제조사의 데이터가 포함되어 있습니다. 문서에는 내부 설계 매뉴얼, 규정 준수 문서, 유지보수 가이드에서 소스를 둔 맥락 기반 답변이 포함되어 있어 기술 및 규제 정보를 정확히 검색할 수 있습니다.

### A1. Playground에서 카탈로그 탐색기로 AI Search를 구성하세요.

Databricks는 이제 AI Playground에서 AI Search 인덱스를 테스트하는 과정을 간소화합니다.

**AI Search 인덱스와 함께 Playground를 실행하려면 다음 단계를 따르세요:**

1. Databricks Workspace에서 **카탈로그 탐색기 (Catalog Explorer)** 를 열고 AI Search 인덱스로 이동하세요.
   - 예시: `{{{catalog_name}}}.{{{schema_name}}}.docs_chunked_index`

1. 인덱스 상세 페이지 오른쪽 상단에서 **Playground에서 시도하기 (Try in Playground)** 버튼을 클릭하세요.

1. AI Playground 사이트는 AI Search 인덱스가 검색 도구로 미리 설정된 상태로 자동으로 열립니다.

1. Playground 인터페이스에서 선호하는 대형 언어 모델(LLM)을 선택하세요. `Claude Sonnet 4.6` 모델을 사용하는 것을 권장합니다.
   * **Use Endpoint**을 클릭하세요.

1. 쿼리를 입력하여 검색과 응답 품질을 테스트하세요.

**팁:** 플레이그라운드에서 수동으로 검색 도구를 추가할 수도 있습니다. 하지만 이 방법은 AI Search이 미리 설정되어 있어서 LLM을 바로 선택하고 실험을 시작할 수 있어 시간을 절약할 수 있습니다.

### A2. 테스트 쿼리와 실험 

AI Search이 설정되었으므로, 이제 다양한 쿼리로 테스트하여 검색 품질과 응답 정확도를 평가해 보겠습니다.

**회수 시스템을 테스트하기 위해 다음 단계를 따르세요:**

1. 채팅 인터페이스에서 지식 기반과 관련된 질문을 입력하세요.
   - 예시: *"오리온 모션 컨트롤러는 고속 이동 중 어떻게 안정성을 유지하는가?"*
   - 예시: *"오리온은 ISO 13849-1로 규정 준수를 어떻게 검증하나요?"*

1. 쿼리를 제출하고 언어 모델의 반응을 관찰하세요.


**모범 사례 실험하기:**

1. **검색된 맥락을 검토하세요:** 어떤 문서나 조각이 검색되었는지 확인하고 그것이 질문과 관련이 있는지 확인하세요.

1. **응답의 질 평가:** 답변이 귀하의 지식 기반을 정확히 반영하고 검색된 문서에 근거하고 있는지 확인하세요.

1. **예외 사례 테스트:** 지식 기반 밖에서 질문하고 다양한 표현을 시도해 견고성을 평가하세요.

1. **반복하고 다듬으세요:** 결과가 좋지 않으면 검색 매개변수를 조정하고, 잘 작동하는 패턴을 노트 확인하세요.


Playground에서 검색 품질에 만족하면, 다음 섹션에서 프로그래매틱하게 에이전트를 구축할 준비가 된 것입니다.

## B. LangChain과 함께 검색 에이전트 구축

이 섹션에서는 이 데모의 범위 내 프레임워크인 **LangChain**를 사용하여 검색 에이전트를 만들 것입니다. LangChain은 언어 모델을 조직 데이터와 연결하는 강력한 도구를 제공하여 문맥 인식 응답과 유연한 에이전트 Workflows을 가능하게 합니다. **여기서는 LangChain을 모범 사례를 보여주기 위해 사용하지만, 대형 언어 모델과 AI Search를 지원하는 다른 프레임워크나 라이브러리에도 유사한 검색 에이전트 패턴을 적용할 수 있습니다.** 개념과 아키텍처는 이전 가능하니, 본인의 생산 요구사항에 가장 적합한 기술을 선택하세요.

우리는 **"agent as code"** 방식을 따르며, 에이전트 구현을 Python 파일(`agent.py`)로 작성합니다. `mlflow`로 모델을 로깅할 때 권장되는 방법입니다. 에이전트는 Unity catalog의 AI Search를 도구로 활용하여 사용자 질문에 답변할 때 동적으로 관련 컨텍스트를 검색할 수 있습니다.

### B1. MLflow 추적 활성화

에이전트 구축을 시작하기 전에, **LangChain**에 대한 MLflow 추적을 활성화해 에이전트의 입력, 도구 사용, 출력을 자세히 관찰할 수 있게 해봅시다.

MLflow는 GenAI Workflows (LangChain을 포함)에 대한 강력한 추적과 관찰 기능을 제공하며, 다양한 프레임워크와 변형을 지원합니다. 이 광범위한 통합은 다양한 환경에서 생성 AI 애플리케이션을 모니터링, 디버깅, 분석할 수 있게 하며, 모두 통합된 MLflow 인터페이스 내에서 이루어집니다.

**💡 참고:** 클래식 compute에서는 MLflow 추적 (`autolog()`)이 기본적으로 활성화되어 있지만, 서버리스 compute에서는 수동으로 활성화해야 합니다.



In [0]:
import mlflow
mlflow.langchain.autolog()

### B2. LangChain 에이전트를 만드세요

In [0]:
llm_endpoint_name = "databricks-claude-sonnet-4-6"

In [0]:
from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langgraph.checkpoint.memory import InMemorySaver


def build_agent(llm_endpoint:str, index_name: str, num_results: int = 3):
    model = ChatDatabricks(
        endpoint=llm_endpoint,
        max_tokens=500,
    )

    vs_tool = VectorSearchRetrieverTool(
        name="orion_knowledge_search",
        index_name=index_name,
        description="Search Orion knowledge base for relevant information",
        num_results=num_results,
    )

    # 선택 사항: 메모리 내 저장소를 사용해 에이전트의 상태를 저장하세요
    checkpointer = InMemorySaver()

    system_prompt = """당신은 오리온 지식 조수(OKA)입니다. 엔지니어와 기술진에게 적합한 명확하고 전문적이며 사실적인 어조로 답변하세요. 오리온 내부 문서에서 검증된 정보만 사용하고, 가능한 경우 출처 참고 자료를 포함하세요. 답을 찾지 못하면 명확히 말하고 관련 섹션이나 다음 단계를 제안하세요. 추측하거나 가정하거나 제공된 맥락 밖의 정보를 제공하지 마십시오."""

    agent = create_agent(
        model=model, 
        tools=[vs_tool], 
        system_prompt=system_prompt,
        checkpointer=checkpointer,
        )
    return agent

# `thread_id` 는 특정 대화에 대한 고유 식별자입니다.
config = {"configurable": {"thread_id": "databricks-demo-4"}}

# 빠른 스모크 테스트
agent = build_agent(llm_endpoint_name, vs_index_name, 3)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is Orion?"}]},
    config=config
)
print(response['messages'][-1].content)


### B3. MLflow 추적 UI 검토

MLflow 추적 UI는 에이전트의 실행 및 도구 사용에 대한 포괄적인 뷰를 제공합니다. 위 출력물은 추적 UI를 보여줍니다. 또는 이전에 진행한 실험의 흔적을 검토하고 싶다면, **실험 (Experiments)** 페이지에서 볼 수 있습니다.
실험의 흔적을 확인하려면 실험을 선택하고 실험 내 **트레이스 (Traces)** 탭으로 이동하세요.

- **요약 (Summary)** 탭는 각 트레이스에 대한 입력, 출력 및 트레이스 메타데이터를 포함한 고수준 정보를 표시합니다.
- **세부 사항 및 일정 (Details & Timeline)** 탭는 트레이스의 각 단계를 상세히 보여주며, 모든 LLM 호출, 호출된 도구, 도구에서 반환된 결과, 최종 생성된 출력을 보여줍니다. 이것은 에이전트의 논리와 데이터 흐름을 이해하는 데 도움이 됩니다.
- 왼쪽에서 타임라인 아이콘을 클릭하면 **타임라인 뷰 (timeline view)** 실행을 활성화하여 각 단계의 시간을 시각화할 수 있어 사용자가 병목 현상이나 성능 문제를 쉽게 식별할 수 있습니다.
- 개별 트레이스를 선택하면 오른쪽 패널에 추가 세부 정보가 나타납니다. 오류가 발생하면 오류 메시지와 관련 맥락을 나열한 **이벤트 (Events)** 탭에서 확인할 수 있습니다.


이 탭들을 검토하면 에이전트 동작을 검증하고, 문제를 디버그하며, 명확하고 실행 가능한 인사이트로 에이전트 개발 Workflows을 최적화할 수 있습니다.

## C. 에이전트를 Model Registery에 등록하십시오

이 절에서는 모델을 Model Registery에 기록하는 방법을 보여드리겠습니다. 먼저, 모든 에이전트 코드를 포함하는 파일을 만들어 notebook에서 에이전트 코드를 추상화해야 합니다. 또한, 에이전트 코드는 어떤 환경에서도 실행할 수 있으므로, 설정 파일을 생성할 `.yaml` 것입니다. 이 기록은 에이전트 코드와 함께 로그됩니다.

### C1. `agent-config` 생성

In [0]:
import yaml

def create_config(llm_endpoint_name: str, index_name: str, num_results: int = 3):
    """Create a minimal YAML config for the agent."""
    config = {
        "llm_endpoint_name": llm_endpoint_name,
        "vector_search": {
            "index_name": index_name,
            "num_results": num_results
        }
    }
    return config


# 설정 파일 생성
llm_endpoint_name = "databricks-claude-sonnet-4-6"

agent_config = create_config(llm_endpoint_name, vs_index_name)

# YAML 파일을 작성하세요 (agent.py이 나중에 읽을 수 있도록)
with open("agent-config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(agent_config, f, sort_keys=False)

print("✅ Config file written: agent-conf.yaml")
print(yaml.safe_dump(agent_config, sort_keys=False))


### C2. 파일에 에이전트 코드를 작성합니다

에이전트 로그에 사용할 `agent.py` 파일을 생성합니다. 이 파일에는 다음이 포함됩니다:
- 설정 파일 로딩 
- LangChain 이전 단계에서 만든 코드
- MLflow API 기반 예측 및 응답 형식


In [0]:
%%writefile agent.py
import os
from uuid import uuid4
from typing import Any, Dict, List

import yaml
import mlflow
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse

from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langgraph.checkpoint.memory import InMemorySaver

# Load agent configuration from YAML file
def _load_config(path: str = "agent-config.yaml") -> Dict[str, Any]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Config file not found at '{path}'")
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    llm_endpoint = cfg.get("llm_endpoint_name")
    vs = cfg.get("vector_search", {}) or {}
    index_name = vs.get("index_name")
    num_results = int(vs.get("num_results", 3))
    if not llm_endpoint or not index_name:
        raise ValueError("Missing 'llm_endpoint_name' or 'vector_search.index_name' in agent-config.yaml")
    return {
        "llm_endpoint_name": llm_endpoint,
        "vs_index_name": index_name,
        "vs_num_results": num_results,
    }

# Build LangChain agent with LLM and AI Search tool
def build_agent(llm_endpoint: str, index_name: str, num_results: int = 3):
    model = ChatDatabricks(endpoint=llm_endpoint, max_tokens=500)
    vs_tool = VectorSearchRetrieverTool(
        name="orion_knowledge_search",
        index_name=index_name,
        description="Search Orion knowledge base for relevant information",
        num_results=num_results,
    )
    checkpointer = InMemorySaver()
    system_prompt = (
        "You are the Orion Knowledge Assistant (OKA). Respond in a clear, professional, and factual tone "
        "appropriate for engineers and technical staff. Use only verified information from Orion's internal "
        "documents, and include source references when available. If the answer cannot be found, clearly state "
        "that and suggest related sections or next steps. Do not speculate, make assumptions, or provide "
        "information outside the provided context."
    )
    agent = create_agent(
        model=model,
        tools=[vs_tool],
        system_prompt=system_prompt,
        checkpointer=checkpointer,
    )
    return agent

# Extract last user message from conversation
def _last_user_text(messages: List[Dict[str, Any]]) -> str:
    user_msgs = [m for m in messages if (m.get("role") == "user")]
    return str(user_msgs[-1].get("content", "")) if user_msgs else str(messages[-1].get("content", ""))

# MLflow ResponsesAgent implementation for LangChain agent
class LangChainResponsesAgent(ResponsesAgent):
    def __init__(self):
        cfg = _load_config()
        self._cfg = cfg
        self._agent = build_agent(
            llm_endpoint=cfg["llm_endpoint_name"],
            index_name=cfg["vs_index_name"],
            num_results=cfg["vs_num_results"],
        )

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        msgs = [m.model_dump() for m in request.input]  # [{'role': 'user'|'assistant', 'content': '...'}, ...]
        _ = _last_user_text(msgs) if msgs else ""

        # Generate a unique thread ID for each prediction
        thread_id = f"oka-{uuid4()}"

        result = self._agent.invoke(
            {"messages": msgs},
            config={"configurable": {"thread_id": thread_id}},
        )
        # Extract agent response text
        try:
            text = result["messages"][-1].content
        except Exception:
            text = str(result)

        return ResponsesAgentResponse(
            output=[self.create_text_output_item(text, str(uuid4()))],
            custom_outputs=request.custom_inputs,
        )

# Set the model for mlflow. This is needed when using agent-as-code approach
AGENT = LangChainResponsesAgent()
mlflow.models.set_model(AGENT)

### C3. 파일에서 에이전트 가져오기

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
from agent import AGENT as agent

mlflow.langchain.autolog()

response  = agent.predict(
    {"input": [{"role": "user", "content": "What is Orion?"}]}
)


## D. 모델을 Model Registery로 기록하고 등록하기

에이전트를 로드하고 테스트한 후에는 에이전트를 모델링 레지스터리로 로그할 수 있습니다. 이를 통해 모델을 버전 변경하고, 별칭을 부여하며, 태그를 하고 권한을 관리할 수 있습니다. Model Registery에서 model serving을 사용해 모델을 배포할 수 있습니다. 참고 에이전트 배치는 이 모듈의 범위에 포함되지 않는다는 점입니다.


### D1. 모델 리소스 정의

에이전트를 MLflow에 로그하기 전에 에이전트가 의존하는 **리소스**를 정의해야 합니다. 리소스는 AI Search 인덱스, 엔드포인트, 테이블, 또는 추론 과정에서 에이전트가 사용하는 함수와 같은 외부 의존성을 나타냅니다.

이 리소스들을 명시적으로 선언함으로써 MLflow는 다음과 같은 조치를 취할 수 있습니다:
* 반복성과 계통을 위해 **의존성을 추적**하세요.
* 배포 전에 필요한 리소스의 **가용성을 검증**하세요.
* **적절한 권한 설정**과 생산 환경에서 접근 제어를 보장합니다.

오리온 지식 어시스턴트를 위해 두 가지 키 리소스를 정의하겠습니다:
1. **DatabricksVectorSearchIndex**: 오리온의 지식 기반을 포함한 AI Search 인덱스.
1. **DatabricksServingEndpoint**: LLM endpoint 응답 생성에 사용됩니다.

이러한 리소스는 모델이 배포될 때 필요한 인프라 구성 요소에 접근할 수 있도록 보장합니다.

**🚨 중요:** 진행하기 전에 교실 설정 설정 파일을 가져왔는지 확인하세요. 이 단계는 이전 섹션에서 Python 커널이 재시작되었고, 올바른 리소스 구성을 위해 카탈로그 및 스키마 이름을 다시 불러와야 하기 때문에 필요합니다.

In [0]:
%run ../Includes/Classroom-Setup-Common

In [0]:
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint

# Python 재시작 후 변수 재정의
vs_index_name = f"{catalog}.{schema}.docs_chunked_index"
llm_endpoint_name = "databricks-claude-sonnet-4-6"

# 에이전트가 의존하는 자원을 정의하세요
resources = [
    DatabricksVectorSearchIndex(index_name=vs_index_name),
    DatabricksServingEndpoint(endpoint_name=llm_endpoint_name)
]

print("Resources defined:")
for resource in resources:
    print(f"  - {resource}")

### D2. 로그 에이전트 모델 MLflow

에이전트의 리소스를 정의했으므로, 이제 MLflow를 사용하여 **에이전트를 모델로 기록**하겠습니다. 로깅은 에이전트 코드, 구성, 의존성 및 메타데이터를 구조화된 형식으로 캡처하여 버전 관리, 추적 및 배포가 가능합니다.

MLflow로 모델을 로그하면, 다음을 기록하는 **실행**을 만듭니다:
* **모델 아티팩트**: 에이전트 코드(`agent.py`)및 구성(`agent-config.yaml`).
* **의존성**: 에이전트를 실행하는 데 필요한 Python 패키지.
* **리소스**: AI Search 인덱스와 서비스 엔드포인트 같은 외부 의존성도 있습니다.
* **입출력 예시**: 예상되는 모델 인터페이스를 보여주는 샘플 데이터.
* **메타데이터 및 태그**: 모델 버전, 목적 및 계보에 관한 정보.

이 기록된 모델은 동일한 의존성과 자원을 가진 어떤 환경에서도 로드, 테스트 및 배포가 가능한 **재현 가능한 산출물**이 됩니다.

In [0]:
import mlflow
from importlib.metadata import version as get_version

# 모델명과 태그
model_name = "orion_knowledge_assistant"
tags_to_register = {
    "model_type": "retrieval_agent",
    "framework": "langchain",
    "use_case": "orion_knowledge_base"
}

# 모델 서명에 대한 입력 예제를 생성하세요
input_example = {
    "input": [
        {"role": "user", "content": "What is Orion?"}
    ]
}

# MLflow 실행을 시작해서 모델을 로그하세요
with mlflow.start_run():
    mlflow.set_tags(tags_to_register)
    
    logged_agent_info = mlflow.pyfunc.log_model(
        name=model_name,
        python_model="agent.py",
        code_paths=["agent-config.yaml"],
        input_example=input_example,
        pip_requirements=[
            f"databricks-vectorsearch=={get_version('databricks-vectorsearch')}",
            f"databricks-langchain=={get_version('databricks-langchain')}",
            f"langchain=={get_version('langchain')}",
            f"mlflow=={get_version('mlflow')}",
        ],
        resources=resources
    )
    
    # 모델 URI는 나중에 사용하기 위해 저장하세요
    model_uri = logged_agent_info.model_uri
    
print(f"✅ Model logged successfully!")
print(f"Model URI: {model_uri}")

### D3. 모델을 Unity Catalog에 등록하세요

이제 에이전트 모델을 기록했으니, **그것을 Unity Catalog의 Model Registry에 등록**하겠습니다. 기록과 등록이 비슷해 보일 수 있지만, MLOps 라이프사이클에서는 서로 다른 목적을 수행합니다.

**차이점 이해하기:**

* **Logging**은 MLflow experiment 실행 내에서 버전 관리 아티팩트를 생성합니다. 그것은 특정 시점의 모델 코드, 의존성 및 메타데이터를 캡처합니다. 기록된 모델은 개별 실행에 연동되어 주로 실험과 개발에 사용됩니다.

* **등록**은 기록된 모델을 Model Registry로 승격시켜 이는 Unity Catalog에서 고유한 이름을 가진 **관리되고 거버넌스된 자산**이 됩니다. 등록된 모델은 다음을 지원합니다:
  - **버전 관리**: 같은 모델의 여러 버전을 추적하세요.
  - **별칭**: 특정 버전에 `Champion` 또는 `Challenger` 같은 라벨을 붙이세요.
  - **거버넌스**: 권한을 적용하고, 태그, 계보 추적을 적용하세요.
  - **배치**: 레지스트리에서 프로덕션 엔드포인트로 모델을 직접 서비스하세요.

에이전트를 Unity Catalog에 등록함으로써 그것을 실험적인 산출물에서 조직 전반에 걸쳐 발견, 관리, 배포할 수 있는 생산 준비가 가능한 자산으로 전환시킵니다.

In [0]:
# 레지스트리 URI를 Unity Catalog로 설정하세요
mlflow.set_registry_uri("databricks-uc")

# Unity Catalog에서 완전 적격 모델 이름을 정의하세요
UC_MODEL_NAME = f"{catalog}.{schema}.orion_knowledge_assistant"

# 모델을 Unity Catalog에 등록하세요
uc_registered_model_info = mlflow.register_model(
    model_uri=model_uri, 
    name=UC_MODEL_NAME
)

print(f"✅ Model registered successfully to Unity Catalog!")
print(f"Model Name: {UC_MODEL_NAME}")
print(f"Version: {uc_registered_model_info.version}")

### D4. 테스트 모델 추론

Unity catalog에 에이전트를 등록했으므로 **이제 로드하고 테스트**하여 정상 작동하는지 확인할 수 있습니다. 레지스트리에서 모델을 로드하면 기록된 정확한 버전을 사용하게 되며, 모든 종속성과 구성이 그대로 유지됩니다.

두 가지 접근법을 시연해 보겠습니다:
1. **모델 URI에서 불러오기**: 로깅 단계에서 얻은 URI를 사용하고 있습니다.
1. **Unity Catalog에서 불러오기**: 완전 적격 모델 이름을 사용하기

두 방법 모두 에이전트를 테스트 입력으로 호출하여 AI Search 인덱스에서 관련 컨텍스트를 검색하고 적절한 응답을 생성하는지 검증할 수 있게 합니다.

In [0]:
# 모델을 모델 URI(pyfunc 스타일)에서 불러옵니다
pyfunc_model = mlflow.pyfunc.load_model(model_uri)

# 모델과 함께 기록된 입력 예제를 사용하세요
input_data = pyfunc_model.input_example

print("Input data:")
print(input_data)
print("\n" + "="*50 + "\n")

# 로드된 모델을 사용해 예측을 합니다
result = mlflow.models.predict(
    model_uri=model_uri,
    input_data=input_data,
    env_manager="uv",
)

print("Agent Response:")
print(result)

### D5. Model Registry UI를 탐색해 보세요

에이전트가 Unity Catalog에 성공적으로 등록되면서, 이제 **Model Registry UI를 탐색**하여 모델을 관리, 모니터링, 관리하는 방법을 이해할 수 있습니다. Model Registry는 모델 버전, 계보, 산출물 및 성능을 추적할 수 있는 중앙 인터페이스를 제공합니다.

**등록된 모델을 확인하려면 다음 단계를 따라주세요:**

1. Catalog Explorer에서 당신의 모델로 이동하세요:
   - Databricks 워크스페이스의 **카탈로그 탐색기**로 이동하세요.
   - `{{{catalog_name}}}.{{{schema_name}}}.orion_knowledge_assistant`로 이동하세요.
   - 모델의 최신 버전을 클릭하세요.

1. 네 가지 주요 탭을 탐색해 보세요:

   * **개요 (Overview)**: 이 탭은 모델 버전에 대한 중요한 정보를 보여줍니다:
     - **지표**: 훈련이나 평가 중에 모델에 로그된 모든 메트릭.
     - **활동 로그**: 변경 사항, 업데이트 및 배포의 연대기 로그입니다.
     - **모델 서명**: 예상되는 데이터 형식을 정의하는 입출력 스키마.
     - **버전 정보**: 이 특정 버전에 대한 상세 정보, 제작일과 제작자 등이 포함됩니다.
     - **활성 엔드포인트**: 현재 이 모델 버전을 사용하는 모든 서비스 엔드포인트들.
     - **태그**: 조직 및 발견을 위한 맞춤형 메타데이터 태그.

   * **리니지 (Lineage)**: 모델 버전 추적을 위해 상류 소스와 하류 소비자를 표시합니다.

   * **유물 (Artifacts)**: 모델에 등록된 모든 파일과 의존성을 나열합니다.

   * **트레이스 (Traces)**: 에이전트 호출 및 디버깅에 대한 상세 실행 추적을 표시합니다.

Model Registry UI는 모델의 라이프사이클에 대한 포괄적인 뷰를 제공하여 버전 관리, 계보 추적 및 조직 전반에 걸친 거버넌스 확보를 용이하게 합니다.

## E. 요약

이 데모에서는 Databricks Mosaic AI를 사용하여, 워크플로 전체를 탐구했습니다. **검색 에이전트**를 빌드, 로그, 등록하여 생산용으로 준비했습니다. 우리는 **AI Playground에서 AI Search를 테스트**하는 것으로 시작했고, 이후 맥락 인식 응답을 위해 LLM과 AI Search에 저장된 지식 기반을 결합한 LangChain 기반 에이전트를 구축했습니다. 우리는 관찰 가능성을 위해 **MLflow 추적**을 활용했고, Python 파일과 YAML 구성을 가진 **"agent as code"** 방식을 채택했으며, 명시적인 **자원 의존성**과 함께 에이전트를 MLflow로 로그했습니다. 마지막으로 에이전트를 Unity catalog의 Model Registry에 등록하여 **관리되고 버전 관리되는 자산으로 변환하여 프로덕션 배포 준비**를 완료했습니다.

**핵심 내용:**

* **AI Playground**에서 **AI Search 인덱스를 테스트**하여 코드를 작성하기 전에 검색 품질과 응답 정확성을 검증하세요.
* **MLflow 추적**을 활성화하여 에이전트의 동작, 도구 사용, LLM 호출, 응답 생성 등을 모니터링합니다.
* **'에이전트를 코드로서' 접근법을 채택**하여 논리를 **Python 파일**로 추상화하고 **YAML 구성**을 적용하여 이동성과 유지보수성을 확보하세요.
* **명시적인 자원 의존성을 정의**(AI Search 인덱스, 서빙 엔드포인트 등)하여 모델을 **MLflow**에 로그하세요.
* **모델을 Unity Catalog의 Model Registry에 등록**하여 버전 관리, 거버넌스, 계통 추적, 운영 배포를 가능하게 합니다.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>